<a href="https://colab.research.google.com/github/Moharram-Khaled/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Moharram-Khaled/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Finding 1 — The Freshness Multiplier

The paper reports that the 31–90 day freshness window had the strongest stable
growth-to-decline ratio at 7.88:1. It also reports that 365+ day content that
was refreshed within 30 days showed a 3.2x health increase and 57x more
impressions in this portfolio.

### Methodology question

How was the refreshed group compared with the non-refreshed group, and were
the groups comparable before the refresh? In particular, I would want to know
whether pages selected for refresh already had stronger demand, quality, or
historical performance. If so, the observed difference could partly reflect
selection effects rather than the refresh itself.

## Finding 2 — AI Model Performance

The paper reports that age-controlled OpenAI and Gemini cohorts lead in
different publication windows. The paper therefore treats the result as
exploratory rather than as evidence that one model family universally
outperforms the other.

### Methodology question

After controlling for content age, were the provider cohorts also comparable
in topic, search intent, publishing conditions, and other important factors?
I would also want to know whether the cohort sizes and validation design were
strong enough to support the comparison across age bands. If these factors
differed between providers, the observed differences could reflect cohort
composition rather than model performance.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In Week 5, the model was evaluated using a standard random train/test split.
For this audit, I use a grouped split by client so that the same client does
not appear in both training and test sets.

This is a stricter validation design because content from the same client can
share characteristics that make random splitting optimistic.

I compare the original random-split result with the grouped result using the
same Logistic Regression model, features, target definition, and metrics.

In [10]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
import pandas as pd

HF_TOKEN = userdata.get("HF_TOKEN")

# Download March 2026
march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

# Download April 2026
april_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

march = pd.read_parquet(march_path)
april = pd.read_parquet(april_path)

print("March shape:", march.shape)
print("April shape:", april.shape)

March shape: (9841378, 30)
April shape: (10424730, 30)


In [11]:
# Create current-window features from March
features = (
    march
    .groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        gsc_impressions=("gsc_impressions", "sum"),
        gsc_avg_position=("gsc_avg_position", "mean"),
        gsc_clicks=("gsc_clicks", "sum")
    )
)

# Create future outcome from April
future = (
    april
    .groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        future_avg_position=("gsc_avg_position", "mean")
    )
)

# Match March features with April outcome
model_df = features.merge(
    future,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Model rows:", len(model_df))
print("Columns:", model_df.columns.tolist())

display(model_df.head(10))

Model rows: 331436
Columns: ['client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_avg_position', 'gsc_clicks', 'future_avg_position']


,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,gsc_clicks,future_avg_position
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,0,NaN,0,NaN
1,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,0,NaN,0,NaN
2,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,0,NaN,0,NaN
3,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1,9.000000,0,10.484848
4,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,0,NaN,0,NaN
5,client_0797ff3a1fc9a6a5,content_0317b24cc1ff5c5d,0,NaN,0,NaN
6,client_0797ff3a1fc9a6a5,content_044c54ec4adcc4b2,0,NaN,0,NaN
7,client_0797ff3a1fc9a6a5,content_04c67f3541177192,331,14.129210,2,17.028086
8,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,33,9.225529,0,NaN
9,client_0797ff3a1fc9a6a5,content_07573a1cc2034981,0,NaN,0,NaN


In [12]:
model_df = model_df.dropna(
    subset=[
        "gsc_impressions",
        "gsc_avg_position",
        "gsc_clicks",
        "future_avg_position"
    ]
).copy()

# 1 = future average position improved
# 0 = did not improve
model_df["target"] = (
    model_df["future_avg_position"] < model_df["gsc_avg_position"]
).astype(int)

print("Rows:", len(model_df))

print("\nTarget distribution:")
print(model_df["target"].value_counts())

print("\nTarget proportions:")
print(model_df["target"].value_counts(normalize=True))

Rows: 158549

Target distribution:
target
0    102856
1     55693
Name: count, dtype: int64

Target proportions:
target
0    0.648733
1    0.351267
Name: proportion, dtype: float64


In [13]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

features_cols = [
    "gsc_impressions",
    "gsc_avg_position",
    "gsc_clicks"
]

X = model_df[features_cols]
y = model_df["target"]
groups = model_df["client_hash_id"]

# Fill any remaining missing values
X = X.fillna(0)

# -----------------------------
# 1. Original random split
# -----------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

random_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

random_model.fit(X_train, y_train)

random_pred = random_model.predict(X_test)

random_accuracy = accuracy_score(y_test, random_pred)
random_precision = precision_score(y_test, random_pred, zero_division=0)
random_recall = recall_score(y_test, random_pred, zero_division=0)
random_f1 = f1_score(y_test, random_pred, zero_division=0)

# -----------------------------
# 2. Grouped split by client
# -----------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

group_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

group_model.fit(X_train_group, y_train_group)

group_pred = group_model.predict(X_test_group)

group_accuracy = accuracy_score(y_test_group, group_pred)
group_precision = precision_score(
    y_test_group,
    group_pred,
    zero_division=0
)
group_recall = recall_score(
    y_test_group,
    group_pred,
    zero_division=0
)
group_f1 = f1_score(
    y_test_group,
    group_pred,
    zero_division=0
)

# -----------------------------
# Comparison
# -----------------------------

comparison = pd.DataFrame({
    "Method": [
        "Random split",
        "Grouped split by client"
    ],
    "Accuracy": [
        random_accuracy,
        group_accuracy
    ],
    "Precision": [
        random_precision,
        group_precision
    ],
    "Recall": [
        random_recall,
        group_recall
    ],
    "F1": [
        random_f1,
        group_f1
    ]
})

display(comparison)

print("Random split train rows:", len(X_train))
print("Random split test rows:", len(X_test))

print("Grouped split train rows:", len(X_train_group))
print("Grouped split test rows:", len(X_test_group))

print(
    "Train clients:",
    groups.iloc[train_idx].nunique()
)

print(
    "Test clients:",
    groups.iloc[test_idx].nunique()
)

print(
    "Client overlap:",
    len(
        set(groups.iloc[train_idx])
        & set(groups.iloc[test_idx])
    )
)

,Method,Accuracy,Precision,Recall,F1
0,Random split,0.689212,0.630002,0.279302,0.387023
1,Grouped split by client,0.767506,0.442967,0.249409,0.319133


Random split train rows: 118911
Random split test rows: 39638
Grouped split train rows: 135314
Grouped split test rows: 23235
Train clients: 34
Test clients: 12
Client overlap: 0


In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The model uses March 2026 historical features and predicts whether average search position improves in April 2026. The target is derived from the future April outcome and is not included in the feature set.

The model features are `gsc_impressions`, `gsc_avg_position`, and `gsc_clicks`, all calculated from the March observation window. The April `future_avg_position` field is used only to construct the target.

I also checked the grouped validation design. The training and test sets contain different clients, with zero client overlap. This reduces the risk that the model is evaluated on clients it has already seen during training.

No future-derived target field, product flag, or future-month feature is used as a model input.

In [15]:
# Leakage audit

feature_columns = [
    "gsc_impressions",
    "gsc_avg_position",
    "gsc_clicks"
]

target_column = "target"

future_columns = [
    "future_avg_position"
]

print("=== Feature columns ===")
print(feature_columns)

print("\n=== Target column ===")
print(target_column)

print("\n=== Future-derived columns kept outside features ===")
print(future_columns)

print("\n=== Feature / target overlap ===")
print(set(feature_columns) & {target_column})

print("\n=== Train/Test client overlap ===")

train_clients = set(
    groups.iloc[train_idx]
)

test_clients = set(
    groups.iloc[test_idx]
)

print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(train_clients & test_clients))

print("\n=== Leakage check ===")

if len(set(feature_columns) & set(future_columns)) == 0:
    print("PASS: No future-derived field is used as a feature.")

if len(train_clients & test_clients) == 0:
    print("PASS: No client appears in both train and test.")

=== Feature columns ===
['gsc_impressions', 'gsc_avg_position', 'gsc_clicks']

=== Target column ===
target

=== Future-derived columns kept outside features ===
['future_avg_position']

=== Feature / target overlap ===
set()

=== Train/Test client overlap ===
Train clients: 34
Test clients: 12
Client overlap: 0

=== Leakage check ===
PASS: No future-derived field is used as a feature.
PASS: No client appears in both train and test.


### Failure examples

The grouped model still makes both false-positive and false-negative predictions. These examples show that the three selected signals do not fully explain future ranking movement.

A false positive is a page predicted to improve when it did not. A false negative is a page that improved but the model did not predict improvement. These errors are useful because they show where the current feature set is incomplete.

In [16]:
# Build predictions for the grouped test set

group_test = model_df.iloc[test_idx][
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_avg_position",
        "gsc_clicks",
        "future_avg_position",
        "target"
    ]
].copy()

group_test["predicted"] = group_pred

group_test["correct"] = (
    group_test["target"] == group_test["predicted"]
)

# False positives
false_positives = group_test[
    (group_test["target"] == 0) &
    (group_test["predicted"] == 1)
].copy()

# False negatives
false_negatives = group_test[
    (group_test["target"] == 1) &
    (group_test["predicted"] == 0)
].copy()

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\n=== False positive examples ===")
display(false_positives.head(5))

print("\n=== False negative examples ===")
display(false_negatives.head(5))

False positives: 1592
False negatives: 3810

=== False positive examples ===


,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,gsc_clicks,future_avg_position,target,predicted,correct
29275,client_0fa64a184f18a4a0,content_1d04ab834a0311b5,525,50.834301,0,56.689716,0,1,False
29363,client_0fa64a184f18a4a0,content_2afaf0949995b900,994,33.572181,2,38.980526,0,1,False
29576,client_0fa64a184f18a4a0,content_4a8fbc569b772e5f,108,28.893924,0,31.499167,0,1,False
29636,client_0fa64a184f18a4a0,content_53d2a0e3d8ab0c33,550,40.098902,0,42.548816,0,1,False
29938,client_0fa64a184f18a4a0,content_7cf62a4e6e74253d,1624,31.115529,1,32.213023,0,1,False



=== False negative examples ===


,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,gsc_clicks,future_avg_position,target,predicted,correct
28954,client_0e1acc6cd57b0eba,content_2abf26b922d62f2b,330,7.675690,1,7.441862,1,0,False
28985,client_0e1acc6cd57b0eba,content_6c6fbfa5fb87a71b,2,8.000000,0,0.000000,1,0,False
29007,client_0e1acc6cd57b0eba,content_895642b49897bc84,91,28.292975,0,25.089850,1,0,False
29019,client_0e1acc6cd57b0eba,content_97f37f355cf34c2f,22,20.378788,0,3.875000,1,0,False
29059,client_0e1acc6cd57b0eba,content_d65f50005cb4245d,1,9.000000,0,8.000000,1,0,False


In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*


### Original claim

The Logistic Regression model can predict which content pages will improve their average search position.

### Evidence-based claim

On this dataset, the Logistic Regression model showed directional predictive signal for future position improvement, but its performance was modest. The model achieved an F1 score of 0.319 under a client-grouped split, compared with 0.387 under a random split.

The grouped split is the more conservative validation result because there was zero client overlap between training and test data. Therefore, the result should be treated as decision-support evidence rather than proof that the model will generalize to unseen clients.

The current features explain only part of future ranking movement, and the observed errors show that additional signals may be needed.

In [18]:
# Final model validation summary

final_results = pd.DataFrame({
    "Validation": [
        "Random split",
        "Grouped by client"
    ],
    "Accuracy": [
        random_accuracy,
        group_accuracy
    ],
    "Precision": [
        random_precision,
        group_precision
    ],
    "Recall": [
        random_recall,
        group_recall
    ],
    "F1": [
        random_f1,
        group_f1
    ]
})

display(final_results)

,Validation,Accuracy,Precision,Recall,F1
0,Random split,0.689212,0.630002,0.279302,0.387023
1,Grouped by client,0.767506,0.442967,0.249409,0.319133


## Self-check

- Two research-paper findings and methodology questions are documented constructively.
- The model was evaluated with both a random split and a client-grouped split.
- The grouped split has zero client overlap between training and test data.
- The model uses only March 2026 features and does not use the future target as an input.
- Real false-positive and false-negative examples are shown.
- Claims are written as observed or measured evidence rather than guaranteed model performance.
- The notebook runs from top to bottom without errors.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.